# Builder Node — Unit Tests
Step-by-step testing of the Builder Node.
Run each cell independently to inspect intermediate outputs.

**Pre-requisites before running:**
1. `OPENAI_API_KEY` set in `.env`
2. Qdrant running with the `openapi_reference` collection indexed (run `test_rag.ipynb` first)

In [1]:
# Step 1 — Imports and setup
import json, os
from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE
from nodes.reader import reader_node
from nodes.planner import planner_node
from nodes.extractor import extractor_node
from nodes.reflector import reflector_node
from nodes.validator import validator_node
from nodes.builder import builder_node

logger = get_logger(__name__)
logger.info("Imports OK.")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
2026-03-31 22:46:49 [INFO] __main__: Imports O

Using Device: cpu


In [2]:
# Step 2 — Run full pipeline up to Validator (limited to 2 high-priority sections)
reader_result = reader_node({
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})
logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s).")

planner_result = planner_node({**reader_result, "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": []})
high_priority  = [s for s in planner_result["extraction_plan"]["sections_to_extract"] if s["priority"] == "high"]
test_plan      = {**planner_result["extraction_plan"], "sections_to_extract": high_priority[:2]}
logger.info(f"Planner OK — {len(test_plan['sections_to_extract'])} section(s) selected.")

extractor_result = extractor_node({
    **reader_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": [],
    "extraction_plan": test_plan, "validation_errors": [], "iteration_count": 0,
})
logger.info(f"Extractor OK — {len(extractor_result['raw_rules'])} raw rule(s).")

reflector_result = reflector_node({
    **reader_result, **extractor_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": [],
    "extraction_plan": test_plan,
})
logger.info(f"Reflector OK — {len(reflector_result['reflected_rules'])} reflected rule(s).")

validator_result = validator_node({
    **reader_result, **extractor_result, **reflector_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": [],
    "extraction_plan": test_plan, "iteration_count": 1,
})
logger.info(f"Validator OK — {len(validator_result['validated_rules'])} validated rule(s).")

2026-03-31 22:46:49 [INFO] nodes.reader: Reader Node started.
2026-03-31 22:46:49 [DEBUG] nodes.reader: Parsed 238 relevant sections from main doc.
2026-03-31 22:46:49 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-03-31 22:46:49 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-31 22:46:49 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-31 22:46:49 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-03-31 22:46:49 [INFO] nodes.reader: Reader Node complete.
2026-03-31 22:46:49 [INFO] __main__: Reader OK — 238 section(s).
2026-03-31 22:46:49 [INFO] nodes.planner: Planner Node started.
2026-03-31 22:46:49 [DEBUG] nodes.planner: Sending 238 section(s) to Planner LLM.
2026-03-31 22:46:49 [DEBUG] open

In [3]:
# Step 3 — Loop back up to MAX_ITERATIONS, mirroring the real pipeline behavior
from graph.conditions import should_loop_or_build
from config.settings import MAX_ITERATIONS

all_validated      = list(validator_result["validated_rules"])
current_errors     = validator_result["validation_errors"]
current_reflected  = reflector_result["reflected_rules"]
iteration_count    = 1

while True:
    decision = should_loop_or_build({
        **reader_result,
        "reflected_rules":   current_reflected,
        "validation_errors": current_errors,
        "iteration_count":   iteration_count,
    })
    logger.info(f"Post-validator decision (iter {iteration_count}): '{decision}'")

    if decision != "extractor":
        logger.info("Proceeding to Builder.")
        break

    logger.info(f"Loop-back: re-extracting {len(current_errors)} failed rule(s).")

    ext = extractor_node({
        **reader_result,
        "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": [],
        "extraction_plan": test_plan,
        "validation_errors": current_errors,
        "iteration_count": iteration_count,
    })
    ref = reflector_node({
        **reader_result, **ext,
        "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": [],
        "extraction_plan": test_plan,
    })
    iteration_count += 1
    val = validator_node({
        **reader_result, **ext, **ref,
        "main_doc_path": TSPEC_DATA_TEST_FILE, "auxiliary_doc_paths": [],
        "extraction_plan": test_plan,
        "iteration_count": iteration_count,
    })

    all_validated     += val["validated_rules"]
    current_errors     = val["validation_errors"]
    current_reflected  = ref["reflected_rules"]
    logger.info(f"Iteration {iteration_count} OK — {len(val['validated_rules'])} validated, {len(current_errors)} still failing.")

logger.info(f"Total validated rules going to Builder: {len(all_validated)}")

2026-03-31 22:48:44 [INFO] graph.conditions: Validator result — error rate: 40.0% (2/5 rules), iteration: 1/3
2026-03-31 22:48:44 [WARNING] graph.conditions: Error rate 40.0% exceeds threshold 10%. Looping back to Extractor (iteration 2).
2026-03-31 22:48:44 [INFO] __main__: Post-validator decision (iter 1): 'extractor'
2026-03-31 22:48:44 [INFO] __main__: Loop-back: re-extracting 2 failed rule(s).
2026-03-31 22:48:44 [INFO] nodes.extractor: Extractor Node started.
2026-03-31 22:48:44 [INFO] nodes.extractor: Loop-back iteration 2 — 2 error(s), re-processing 1 section(s).
2026-03-31 22:48:44 [DEBUG] tools.rag_tools: Retrieved 5 chunk(s) for: '12.1.1.1.1 Introduction — Extract overview and mapping princ'
2026-03-31 22:48:44 [DEBUG] nodes.extractor: Extracting [322] '12.1.1.1.1 Introduction' (priority=high).
2026-03-31 22:48:44 [DEBUG] openai._base_client: Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Helper-Method': 'chat.completions.parse', 'X-

In [4]:
# Step 4 — Test builder_node() with accumulated validated rules
builder_state = {
    **reader_result,
    "main_doc_path":      TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":    {**test_plan},
    "validated_rules":    all_validated,
    "validation_errors":  current_errors,
    "iteration_count":    iteration_count,
}

builder_result = builder_node(builder_state)

assert "final_output_path" in builder_result
output_path = builder_result["final_output_path"]
assert os.path.isfile(output_path), f"Output file not found: {output_path}"

logger.info(f"builder_node OK — file saved at:")
logger.info(f"  {output_path}")

2026-03-31 22:49:24 [INFO] nodes.builder: Builder Node started.
2026-03-31 22:49:24 [WARNING] nodes.builder: Max iterations reached — force-including 1 rule(s) that failed validation.
2026-03-31 22:49:24 [DEBUG] nodes.builder:   Force-included [322]: The IS operation modifyMOIAttributes is mapped to the HTTP P
2026-03-31 22:49:24 [INFO] nodes.builder: Builder Node complete — 6 rule(s) saved to '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/outputs/rules_bank/rules_bank_28532-i00_partial_20260331_224924.json'.
2026-03-31 22:49:24 [DEBUG] nodes.builder:   path_operation: 6 rule(s)
2026-03-31 22:49:24 [INFO] __main__: builder_node OK — file saved at:
2026-03-31 22:49:24 [INFO] __main__:   /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/outputs/rules_bank/rule

In [5]:
# Step 5 — Inspect the output JSON
with open(output_path, encoding="utf-8") as f:
    rules_bank = json.load(f)

meta    = rules_bank["metadata"]
summary = rules_bank["summary"]
rules   = rules_bank["rules"]

# Validate top-level structure
assert set(rules_bank.keys()) == {"metadata", "summary", "rules"}

# Validate metadata fields
for field in ("source_document", "generated_at", "model", "total_rules",
              "sections_planned", "sections_with_rules", "iterations_run"):
    assert field in meta, f"Missing metadata field: {field}"
assert meta["total_rules"] == len(rules)

# Validate summary fields
for field in ("total_validated", "final_error_count", "rules_by_type", "rules_by_section"):
    assert field in summary, f"Missing summary field: {field}"
assert summary["total_validated"] == len(rules)

logger.info("Output JSON structure OK.")
logger.info("")
logger.info("=== METADATA ===")
for k, v in meta.items():
    logger.info(f"  {k}: {v}")

logger.info("")
logger.info("=== SUMMARY ===")
logger.info(f"  total_validated  : {summary['total_validated']}")
logger.info(f"  final_error_count: {summary['final_error_count']}")
logger.info(f"  rules_by_type:")
for rtype, count in summary["rules_by_type"].items():
    logger.info(f"    {rtype}: {count}")
logger.info(f"  rules_by_section:")
for sid, count in summary["rules_by_section"].items():
    logger.info(f"    [{sid}]: {count}")

logger.info("")
logger.info("=== RULES ===")
for i, rule in enumerate(rules, start=1):
    logger.info(f"Rule {i} — [{rule['section_id']}] {rule['rule_type']}")
    logger.info(f"  rule_text     : {rule['rule_text']}")
    logger.info(f"  openapi_object: {rule['openapi_mapping']['openapi_object']}")
    logger.info(f"  openapi_field : {rule['openapi_mapping']['openapi_field']}")
    logger.info(f"  openapi_value : {rule['openapi_mapping']['openapi_value']}")
    logger.info(f"  confidence    : {rule.get('confidence', 'n/a')}")
    logger.info("")

2026-03-31 22:49:24 [INFO] __main__: Output JSON structure OK.
2026-03-31 22:49:24 [INFO] __main__: 
2026-03-31 22:49:24 [INFO] __main__: === METADATA ===
2026-03-31 22:49:24 [INFO] __main__:   source_document: /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../../../Dataset/TSpec-LLM/3GPP-clean/Rel-18/28_series/28532-i00.md
2026-03-31 22:49:24 [INFO] __main__:   generated_at: 2026-03-31T22:49:24.069554-03:00
2026-03-31 22:49:24 [INFO] __main__:   model: gpt-4.1-mini
2026-03-31 22:49:24 [INFO] __main__:   total_rules: 6
2026-03-31 22:49:24 [INFO] __main__:   sections_planned: 2
2026-03-31 22:49:24 [INFO] __main__:   sections_with_rules: 1
2026-03-31 22:49:24 [INFO] __main__:   iterations_run: 3
2026-03-31 22:49:24 [INFO] __main__: 
2026-03-31 22:49:24 [INFO] __main__: === SUMMARY ===
2026-03-31 22:49:24 [INFO] __main__:   total_validated  : 6
2026-03-31 22:49:24 [INFO] __main__: 

In [6]:
rules

[{'section_id': '322',
  'section_title': '12.1.1.1.1 Introduction',
  'rule_type': 'path_operation',
  'rule_text': 'The IS operation createMOI is mapped to the HTTP PUT method on the resource URI {MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}.',
  'openapi_mapping': {'openapi_object': 'paths./ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}',
   'openapi_field': 'put',
   'openapi_value': 'PUT'},
  'confidence': 0.8,
  'reasoning': "1. The rule states that the IS operation 'createMOI' is mapped to the HTTP PUT method on a specific resource URI pattern: {MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}. This is a typical pattern in 3GPP specifications where operations are mapped to RESTful methods and URIs. The section title '12.1.1.1.1 Introduction' suggests this is an introductory part of a larger section likely describing the mapping of IS operations to RESTful APIs. Although the exact text of the section is not provided, the rule 